# TUTOR-60 · 1 — The decision, before there is any data worth having

**The situation.** A state has **9.0 million dollars** for one school year and
**12,000 students** reading two or more grade levels behind. Two things are for sale
for those students:

- **structured small-group tutoring**, at 20 USD per weekly minute per student-year;
- **caregiver messaging**, at 9 USD per weekly message per student-year, capped at five
  a week because past that nobody reads them.

The state must decide **how much of each to buy per student** — and therefore, since the
budget is fixed, **how many of the 12,000 it can reach**. A pilot district ran 90 weekly
minutes and liked it. The state cannot afford 90 minutes for everyone and wants to know
whether it should.

This notebook does the part that comes before any experiment: it writes the decision
down as an object, asks what the state's existing data can and cannot answer, declares
the estimand the decision actually needs, and prices the uncertainty. Notebook 2 designs
the trial that uncertainty justifies.

| | reaches into |
|---|---|
| the decision and its objective | `design.DecisionSpec`, `design.ValuePerOutcome` |
| what the observational record supports | `identify.ols`, `identify.CausalGraph`, `identify.identify` |
| the quantity to be estimated | `estimands.Estimand`, `core.Population`, `core.TimeWindow` |
| what the uncertainty is worth | `design.prior_from_history`, `design.evpi_gaussian` |

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import tutoring as T
from axiom.core import D, Intervention, Population, TimeWindow
from axiom.design import (
    DecisionSpec, StudySummary, ValuePerOutcome, evoi_gaussian, evpi_gaussian, prior_from_history,
)
from axiom.estimands import Estimand, Level, Quantity
from axiom.identify import CausalGraph, LinearEstimate, adjustment_sets, identify, ols, requires_unmeasured

from axiom.display import enable

enable();  # every axiom result renders itself from here on

print(f"budget            {T.BUDGET:>12,.0f} USD")
print(f"eligible students {T.N_ELIGIBLE:>12,d}")
print(f"                  {T.BUDGET / T.N_ELIGIBLE:>12,.0f} USD per student if the state reaches all of them")
print(f"tutoring          {T.MINUTE_COST:>12,.0f} USD per weekly minute per student-year")
print(f"messaging         {T.MESSAGE_COST:>12,.0f} USD per weekly message, capped at {T.MESSAGE_CAP_PER_WEEK}")

## 1 · The decision, as an object

The objective is **total reading points delivered to the cohort**, not the effect on a
student who gets the program. Those are different quantities and they disagree, because
every extra dollar per student is a student who is not served at all:

```
cohort points  =  min(eligible, budget / cost_per_student)  x  gain(cost_per_student)
```

Nothing about that formula needs an experiment. It is arithmetic, and it already
constrains the answer: below about 750 USD per student the budget reaches everybody and
the second factor is all that matters; above it, every further dollar trades gain per
student against students served.

In [ ]:
value = ValuePerOutcome(
    value=T.VALUE_PER_POINT, outcome_unit="point", numeraire="USD",
    source="state cost-benefit memorandum 2026: 0.045 sd per point, 31,000 USD per sd",
)
decision = DecisionSpec(
    name="fund_tutoring", threshold=1.5,
    value_per_outcome_unit=value.value * T.N_ELIGIBLE, numeraire="USD",
)
print(value.ledger_line().statement)
print(f"\ndecision '{decision.name}': act if the per-student gain exceeds {decision.threshold} "
      f"{T.OUTCOME_UNIT}\nvalue of one point across the cohort: {decision.value_per_outcome_unit:,.0f} USD")

cost = np.linspace(50.0, 3045.0, 400)
served = T.students_served(cost, 0.0)
fig = T.figure("What the budget buys, before anyone estimates anything",
               "cost per student per year (USD)", "eligible students reached")
fig.add_trace(go.Scatter(x=cost, y=served, line={"color": T.DECISION_COLOR, "width": 3}, name="reached"))
fig.add_hline(y=T.N_ELIGIBLE, line={"dash": "dot", "color": T.TRUTH_COLOR},
              annotation_text="all 12,000 eligible", annotation_position="top right")
fig.add_vline(x=T.BUDGET / T.N_ELIGIBLE, line={"dash": "dot", "color": T.TRUTH_COLOR},
              annotation_text="750 USD — the point where the budget stops binding")
for label, minutes in (("pilot: 90 min/wk", 90), ("30 min/wk", 30)):
    fig.add_annotation(x=minutes * T.MINUTE_COST, y=float(T.students_served(minutes * T.MINUTE_COST, 0.0)),
                       text=label, showarrow=True, arrowhead=2, ay=-30)
fig.show()

The pilot's 90 minutes costs 1,800 USD per student, which reaches **5,000** of the
12,000. Whether that is the right trade depends entirely on the shape of the response
curve between 0 and 150 weekly minutes — which is the thing nobody has measured.

## 2 · What the state already has, and what it is worth

240 schools in the lowest-performing tier reported last year's tutoring intensity and
last year's reading gain. Nobody randomized anything. Schools chose.

In [ ]:
prior = T.prior_year(seed=11)
print(prior[["minutes_per_week", "poverty_share", "gain"]].describe().loc[["mean", "std", "min", "max"]].round(2))

naive: LinearEstimate = ols(prior, y="gain", x="tutoring")
adjusted = ols(prior, y="gain", x="tutoring", covariates=["poverty_share"])
# The average slope of the *true* curve over the range these schools actually used.
lo, hi = float(prior.tutoring.quantile(0.1)), float(prior.tutoring.quantile(0.9))
truth_slope = float((T.tutoring_gain(hi) - T.tutoring_gain(lo)) / (hi - lo))
print(f"\n{'estimator':28s} {'points per 1000 USD':>20s}")
for label, est in (("naive", naive), ("adjusted for poverty share", adjusted)):
    print(f"{label:28s} {est.estimate * 1000:>13.2f} ± {est.se * 1000:.2f}")
print(f"{'the truth, over this range':28s} {truth_slope * 1000:>13.2f}")
print(f"\nthe naive slope is {naive.estimate / truth_slope:.1f}x the truth; "
      f"adjusting for the one proxy in the file leaves {adjusted.estimate / truth_slope:.1f}x")

In [ ]:
fig = T.figure("240 schools, nobody randomized: the slope is not the effect",
               "tutoring", "annualized reading gain (points)", height=430)
fig.add_trace(go.Scatter(
    x=prior.tutoring, y=prior.gain, mode="markers", name="a school, last year",
    marker={"size": 7, "color": prior.resources_unobserved, "colorscale": "RdYlBu", "reversescale": True,
            "line": {"width": 0.5, "color": "white"},
            "colorbar": {"title": "school<br>resources<br>(unobserved)", "thickness": 12, "len": 0.7}},
))
grid = np.linspace(0.0, T.TUTORING_MAX, 200)
fitted = float(prior.gain.mean()) + naive.estimate * (grid - float(prior.tutoring.mean()))
fig.add_trace(go.Scatter(x=grid, y=fitted, name="the regression everyone runs",
                         line={"color": T.DECISION_COLOR, "width": 3}))
fig.add_trace(go.Scatter(x=grid, y=T.TRUTH.baseline_gain + T.tutoring_gain(grid),
                         name="the causal curve", line={"color": T.TRUTH_COLOR, "width": 3, "dash": "dash"}))
T.minutes_axis(fig)
fig.show()

The colour is the confounder. Better-resourced schools sit up and to the right — they
bought more tutoring *and* would have gained more without it. The regression line runs
through that diagonal and reports a slope two and a half times the truth, and adjusting
for the one proxy in the file barely moves it.

Two other things this picture says. The true curve **turns over** past about 90 weekly
minutes, where the tutoring block starts displacing more instruction than it adds — a
straight line through this cloud cannot see that, and the decision is made near the turn.
And almost no school in the state has ever run more than about 115 minutes, so on the
part of the range the pilot is arguing about, the observational record is not merely
confounded: it is empty.

## 3 · The graph says so, before the data does

The bias above is not a small-sample problem and no amount of care with the columns in
the file fixes it. `axiom.identify` says why: write the graph down and ask.

In [ ]:
graph = CausalGraph.from_edges(
    "resources -> poverty_share, resources -> tutoring, poverty_share -> tutoring, "
    "resources -> gain, poverty_share -> gain, tutoring -> gain",
    unmeasured=["resources"], name="what_the_district_knows",
)
verdict = identify(graph, "tutoring", "gain")
print("status                :", verdict.status)
print("route                 :", verdict.route)
print("adjustment sets        :", adjustment_sets(graph, "tutoring", "gain"))
print("needs an unmeasured node:", requires_unmeasured(graph, "tutoring", "gain"))
print("\nassumptions it would take to believe the observational number:")
for assumption in verdict.verdict.assumptions:
    print(f"  [{assumption.state}] {assumption.name}: {assumption.statement}")

`status='downgraded'`, route `backdoor_unmeasured`, and **no admissible adjustment set
exists**. The path `tutoring <- resources -> gain` stays open whatever is conditioned on,
because `resources` is not in the file. Poverty share is a proxy for it, and a proxy
blocks a path only if it *is* the confounder.

That verdict is the entire argument for spending money on a trial. It is also the reason
the trial has to randomize: randomization deletes both arrows into `tutoring` and leaves
a graph with nothing to adjust for.

In [ ]:
randomized = CausalGraph.from_edges(
    "resources -> poverty_share, resources -> gain, poverty_share -> gain, tutoring -> gain",
    unmeasured=["resources"], name="what_a_trial_would_know",
)
trial_verdict = identify(randomized, "tutoring", "gain")
print("after randomization   :", trial_verdict.status, "| route", trial_verdict.route,
      "| adjust for", trial_verdict.adjustment_set)

## 4 · The estimand the decision needs

Two quantities, and it matters which one is written down.

1. The **contrast** at a given allocation: what one served student gains, relative to
   business as usual. This is what a trial estimates.
2. The **cohort objective**: total points across everyone the budget reaches. This is
   what the decision maximizes, and it is a *function of the whole curve* rather than
   of any one contrast — which is why the trial in notebook 2 is a dose-response design
   and not a two-arm comparison.

The estimand is declared at the **level of assignment** (the school), over the **school
year**, for the eligible population, and against the **assigned** allocation, not the
delivered one — schools will not deliver every minute they are given, and delivery is a
consequence of assignment, so conditioning on it would undo the randomization.

In [ ]:
eligible = T.eligible_population()
school_year = T.school_year()
tutoring_entity, messaging_entity = T.treatments()

contrast = Estimand(
    name="growth_at_pilot_allocation",
    quantity=Quantity(kind="contrast"),
    treatment=tutoring_entity,
    intervention=Intervention(doses={"tutoring": 1800.0, "messaging": T.MESSAGING_MAX}, version="assigned"),
    reference=Intervention(doses={"tutoring": 0.0, "messaging": 0.0}, version="assigned"),
    outcome=T.outcome(),
    population=eligible,
    window=school_year,
    level=Level(unit="cluster", interference="none"),
    dimension=D.outcome,
    description=(
        "annualized reading growth for a student in a school assigned the pilot allocation "
        "(90 weekly tutoring minutes and 5 weekly caregiver messages), against no program"
    ),
)
print("estimand      :", contrast.name)
print("content hash  :", contrast.content_hash()[:16])
print("intervention  :", contrast.intervention.doses, f"({contrast.intervention.version})")
print("level         :", contrast.level.unit, "| interference:", contrast.level.interference)
print("window        :", contrast.window)

recommended = contrast.model_copy(update={
    "name": "growth_at_720", "intervention":
    Intervention(doses={"tutoring": 720.0, "messaging": T.MESSAGING_MAX}, version="assigned")})
print("\nthe same estimand at a different allocation differs in exactly one facet:",
      contrast.differing_facets(recommended))

## 5 · What is believed now

Two small studies exist: a single-district pilot two years ago and a charter-network
evaluation from six years ago, both reporting the contrast at roughly the pilot
allocation. `prior_from_history` discounts them by age and combines them.

In [ ]:
history = [
    StudySummary(name="charter_network_2020", treatment="tutoring", estimate=3.9, se=1.4,
                 periods_ago=6.0, definition="wald"),
    StudySummary(name="district_pilot_2024", treatment="tutoring", estimate=5.8, se=1.9,
                 periods_ago=2.0, definition="wald"),
]
prior_mean, prior_sd = prior_from_history(history, half_life_periods=5.0)
print(f"prior on the contrast at the pilot allocation: {prior_mean:.2f} ± {prior_sd:.2f} {T.OUTCOME_UNIT}")
print(f"the truth, for the record: {float(T.mean_gain(1800.0 * T.TRUTH.fidelity_mean, T.MESSAGING_MAX)):.2f}")

x = np.linspace(-1.0, 12.0, 400)
density = np.exp(-0.5 * ((x - prior_mean) / prior_sd) ** 2) / (prior_sd * np.sqrt(2 * np.pi))
fig = T.figure("What is believed before the trial", f"contrast at the pilot allocation ({T.OUTCOME_UNIT})", "density")
fig.add_trace(go.Scatter(x=x, y=density, fill="tozeroy", fillcolor=T.rgba(T.TUTORING_COLOR, 0.22),
                         line={"color": T.TUTORING_COLOR, "width": 2.5}, name="prior from two studies"))
for study in history:
    fig.add_vline(x=study.estimate, line={"dash": "dot", "color": T.ACCENT},
                  annotation_text=study.name, annotation_position="top")
fig.add_vline(x=decision.threshold, line={"color": T.DECISION_COLOR, "width": 2},
              annotation_text="decision threshold", annotation_position="bottom left")
fig.show()

The prior is wide, centred well above the threshold, and — crucially — says nothing at
all about the *shape* between 0 and 150 minutes. Both studies report one allocation. The
decision needs the curve.

## 6 · What resolving it is worth — and which uncertainty is the expensive one

`evpi_gaussian` prices perfect information against a prior and a decision: the expected
cost of choosing under uncertainty rather than knowing. Run it on the go/no-go decision
first.

In [ ]:
evpi_go = evpi_gaussian(decision, prior_mean, prior_sd)
print(f"EVPI on 'is the program worth running at all': {evpi_go:,.0f} {decision.numeraire}")
print(f"  the prior sits {(prior_mean - decision.threshold) / prior_sd:.1f} prior sds above the threshold, "
      f"so P(the trial changes this decision) is about {2 * (1 - 0.9986):.1%}")

**A hundred and thirty thousand dollars**, against a trial that will cost seven figures.
That is the right answer, not a small one: the prior sits two standard deviations clear
of the threshold, so almost no trial result would talk the state out of running *a*
program. If go/no-go were the decision, the honest recommendation would be to skip the
trial and start tutoring.

Go/no-go is not the decision. **The dose is.** And the dose decision is not a threshold
on one contrast — it is a comparison between two allocations the state could fund, whose
difference the prior says almost nothing about, because both historical studies report
the same allocation and neither reports a curve.

Price *that*. Draw from the prior on the pilot allocation, put a prior on how much of
that gain survives at a third of the dose — the state's own reading experts were asked
and split between "the curve has mostly saturated by half an hour" and "dose is close to
proportional", which is a prior spanning roughly 0.15 to 0.90 — and look at the
difference in **cohort points** between funding 90 minutes for whoever the budget
reaches and funding 35 minutes for everyone.

In [ ]:
rng = np.random.default_rng(4)
n_draws = 40_000
gain_at_pilot = rng.normal(prior_mean, prior_sd, size=n_draws)
# What fraction of the pilot's gain survives at 35 weekly minutes? Nobody knows: somewhere
# between "most of it" (the curve saturates early) and "a sixth" (dose is proportional).
surviving = rng.beta(2.0, 2.0, size=n_draws) * 0.75 + 0.15
pilot_points = T.students_served(1800.0, T.MESSAGING_MAX) * gain_at_pilot
low_points = T.students_served(700.0, T.MESSAGING_MAX) * gain_at_pilot * surviving
difference = low_points - pilot_points

dose_decision = DecisionSpec(name="prefer_the_low_allocation", threshold=0.0,
                             value_per_outcome_unit=value.value, numeraire="USD")
evpi_dose = evpi_gaussian(dose_decision, float(difference.mean()), float(difference.std()))
print(f"difference in cohort points (35 min for everyone - 90 min for 5,000):")
print(f"  {difference.mean():>12,.0f} points on average, sd {difference.std():,.0f}")
print(f"  P(the low allocation is better) = {(difference > 0).mean():.2f}")
print(f"  the two are equal when {100 * float(T.students_served(1800.0, T.MESSAGING_MAX) / T.students_served(700.0, T.MESSAGING_MAX)):.0f}% "
      f"of the pilot's gain survives at 35 minutes -- which is exactly what nobody knows")
print(f"\nEVPI on the dose decision: {evpi_dose:,.0f} {dose_decision.numeraire}")
print(f"  {evpi_dose / max(evpi_go, 1.0):,.0f}x the value of resolving go/no-go")
for se_points in (12_000.0, 6_000.0, 3_000.0, 1_500.0):
    result = evoi_gaussian(dose_decision, float(difference.mean()), float(difference.std()), se_points)
    print(f"  a trial that pins the difference to +-{se_points:>7,.0f} points -> "
          f"EVSI {result.evsi:>11,.0f}  ({result.evsi / evpi_dose:5.1%} of perfect)")

In [ ]:
fig = T.figure("The expensive uncertainty is the dose, not the go/no-go",
               "cohort points, 35 min for everyone minus 90 min for 5,000", "density", height=400)
counts, edges = np.histogram(difference, bins=70, density=True)
centres = 0.5 * (edges[1:] + edges[:-1])
fig.add_trace(go.Scatter(x=centres, y=counts, fill="tozeroy", fillcolor=T.rgba(T.ACCENT, 0.22),
                         line={"color": T.ACCENT, "width": 2}, name="prior on the difference"))
fig.add_vline(x=0.0, line={"color": T.DECISION_COLOR, "width": 2},
              annotation_text="the two allocations are equal here", annotation_position="top left")
fig.show()

## What notebook 1 established

1. **The objective is a cohort total, not an effect size.** `min(eligible, budget/cost) x
   gain(cost)` is arithmetic the state already owns, and it means the decision is made on
   the *shape* of a dose-response curve rather than on any single contrast.
2. **The observational record cannot answer it.** No admissible adjustment set exists —
   `identify` returns `downgraded` on the `backdoor_unmeasured` route — and the naive
   slope is two and a half times the truth, in the direction that flatters the program.
3. **The estimand is declared**, at the school level, against the *assigned* allocation,
   with a content hash, before anybody has seen a trial number.
4. **The expensive uncertainty is not the one everybody argues about.** Resolving
   go/no-go is worth 131,000 USD, because the prior is already two standard deviations
   clear of the threshold. Resolving the *dose* is worth 2.3 million — seventeen times as
   much — because the two candidate allocations are separated by whether 41 % of the
   pilot's gain survives at a third of the dose, and nobody knows. A trial designed to
   answer the first question would be a waste of money; notebook 2 designs one for the
   second.

In [ ]:
print("carried into notebook 2:")
carry = {
    "prior_mean": prior_mean, "prior_sd": prior_sd,
    "evpi_go_no_go": float(evpi_go), "evpi_dose": float(evpi_dose),
    "decision": decision.to_json(), "estimand": contrast.to_json(),
}
for key, item in carry.items():
    print(f"  {key:16s}", f"{item:,.2f}" if isinstance(item, float) else "<spec>")